# v10 Report Figures

Generates Fig 2, Fig 3, Fig 4 for `report/v10/report-v10.md`.

- **Fig 2** — 2×2 AP and Faithfulness heatmaps (skip × level)
- **Fig 3** — Pixel vs Patch Faithfulness bar chart (all 4 core models)
- **Fig 4** — AP vs U-Net upper bound (full model family)

All figures saved to `report/v10/figures/`.

In [1]:
import sys, os
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

FIG_DIR = Path('report/v10/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Shared style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.dpi': 150,
})
print('Output dir:', FIG_DIR.resolve())

Output dir: /Users/amo/programData/cardiac-proto-segmentation/report/v10/figures


## Fig 2 — 2×2 Heatmaps: AP and Faithfulness

In [2]:
# 2×2 matrix data
# rows: [With skip, No skip], cols: [L3+L4, L4 only]
ap_matrix = np.array([
    [0.051, 0.057],   # with skip
    [0.301, 0.312],   # no skip
])
faith_matrix = np.array([
    [0.069, 0.093],   # with skip
    [0.035, 0.012],   # no skip
])
row_labels = ['With skip\n(ProtoSegNet)', 'No skip\n(ProtoSegNetV2)']
col_labels = ['L3+L4', 'L4 only']
model_names = [
    ['Stage 29\n(0.051)', 'Stage 8A\n(0.057)'],
    ['9b\n(0.301)',       '9a\n(0.312)'],
]
faith_names = [
    ['Stage 29\n(0.069)', 'Stage 8A\n(0.093)'],
    ['9b\n(0.035)',       '9a\n(0.012)'],
]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

for ax, matrix, names, title, cmap, vmin, vmax in [
    (axes[0], ap_matrix,    model_names, 'Average Precision (AP)',           'Blues',  0.0, 0.35),
    (axes[1], faith_matrix, faith_names, 'Faithfulness (pixel-level)',        'Oranges', 0.0, 0.12),
]:
    im = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(col_labels, fontsize=11, fontweight='bold')
    ax.set_yticks([0, 1])
    ax.set_yticklabels(row_labels, fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=13, pad=10)

    # Annotate cells
    for i in range(2):
        for j in range(2):
            val = matrix[i, j]
            # Choose text color based on cell brightness
            brightness = (val - vmin) / (vmax - vmin)
            txt_color = 'white' if brightness > 0.55 else 'black'
            ax.text(j, i, names[i][j], ha='center', va='center',
                    fontsize=10.5, color=txt_color, fontweight='bold')

    # Add border to best cell
    best_idx = np.unravel_index(np.argmax(matrix), matrix.shape)
    rect = plt.Rectangle(
        (best_idx[1]-0.5, best_idx[0]-0.5), 1, 1,
        linewidth=2.5, edgecolor='gold', facecolor='none'
    )
    ax.add_patch(rect)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=9)

# Add axis labels
axes[0].set_xlabel('Prototype Level', fontsize=11)
axes[0].set_ylabel('Decoder Architecture', fontsize=11)
axes[1].set_xlabel('Prototype Level', fontsize=11)

# Shared subtitle
fig.suptitle(
    'Fig 2  |  Two-Barrier Effect: AP improves 6× removing skip; Faithfulness worsens',
    fontsize=12, y=1.02, fontweight='bold'
)

# Arrows / annotations
for ax, arrow_txt in [(axes[0], '←  Bypass removed\n   AP ×6'), (axes[1], '←  Bypass removed\n   Faith. ÷2')]:
    ax.annotate('', xy=(0, 1.45), xytext=(0, 0.55),
                xycoords='data', textcoords='data',
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

fig.tight_layout()
out_path = FIG_DIR / 'fig2_2x2_heatmap.png'
fig.savefig(out_path, bbox_inches='tight', dpi=150)
print(f'Saved: {out_path}')
plt.close()

Saved: report/v10/figures/fig2_2x2_heatmap.png


## Fig 3 — Pixel vs Patch Faithfulness

In [3]:
models    = ['Stage 29\n(skip, L3+L4)', 'Stage 8A\n(skip, L4)', '9b\n(no-skip, L3+L4)', '9a\n(no-skip, L4)']
pixel_f   = [0.069, 0.093, 0.035, 0.012]
patch_f   = [0.212, 0.161, 0.200, 0.259]
ratios    = [p/q for p, q in zip(patch_f, pixel_f)]

x     = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4.5))

bars1 = ax.bar(x - width/2, pixel_f, width, label='Pixel-level Faith.',
               color='#5B9BD5', alpha=0.9, zorder=3)
bars2 = ax.bar(x + width/2, patch_f, width, label='Patch-level Faith. (bs=16)',
               color='#ED7D31', alpha=0.9, zorder=3)

# Annotate ratios above patch bars
for i, (bar, ratio) in enumerate(zip(bars2, ratios)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.006,
            f'{ratio:.1f}×', ha='center', va='bottom', fontsize=9.5,
            color='#8B4513', fontweight='bold')

# Annotate pixel values
for bar, val in zip(bars1, pixel_f):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8.5, color='#2F5597')

# Annotate patch values
for bar, val in zip(bars2, patch_f):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.013,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8.5, color='#8B4513')

# Highlight the 9a column with an annotation box
ax.annotate('21× lift\n(geometry,\nnot training)',
            xy=(x[3] + width/2, patch_f[3]),
            xytext=(x[3] + 0.55, patch_f[3] - 0.05),
            fontsize=9, color='#C00000',
            arrowprops=dict(arrowstyle='->', color='#C00000', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.3', fc='#FFE6E6', ec='#C00000', lw=1))

# Dashed line at 0.15 gate
ax.axhline(0.15, color='gray', linestyle='--', lw=1, alpha=0.6, label='Gate: 0.15')
ax.text(3.6, 0.152, 'gate (0.15)', fontsize=8.5, color='gray', va='bottom')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10)
ax.set_ylabel('Faithfulness', fontsize=11)
ax.set_ylim(0, 0.31)
ax.set_title('Fig 3  |  Pixel vs Patch Faithfulness — Resolution Barrier diagnosis',
             fontsize=12, fontweight='bold', pad=10)
ax.legend(fontsize=10, loc='upper left')
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Color the no-skip model labels
for i, tick in enumerate(ax.get_xticklabels()):
    tick.set_color('#C00000' if i >= 2 else '#2F5597')

skip_patch   = mpatches.Patch(color='#2F5597', label='With skip (ProtoSegNet)')
noskip_patch = mpatches.Patch(color='#C00000', label='No skip (ProtoSegNetV2)')
leg2 = ax.legend(handles=[skip_patch, noskip_patch], fontsize=9,
                 loc='upper right', title='Architecture')
ax.add_artist(ax.legend(fontsize=10, loc='upper left'))
ax.add_artist(leg2)

fig.tight_layout()
out_path = FIG_DIR / 'fig3_pixel_vs_patch_faith.png'
fig.savefig(out_path, bbox_inches='tight', dpi=150)
print(f'Saved: {out_path}')
plt.close()

Saved: report/v10/figures/fig3_pixel_vs_patch_faith.png


## Fig 4 — AP vs U-Net Upper Bound (full model family)

In [4]:
# Full model family, sorted by AP
data = [
    # (label, AP, Dice, is_unet, skip)
    ('Stage 29\n(skip, L3+L4)',     0.051, 0.821, False, True),
    ('Stage 8A\n(skip, L4)',        0.057, 0.810, False, True),
    ('Stage 8 Full\n(skip, L1–L4)', 0.102, 0.817, False, True),
    ('Stage 34b\n(skip, ALC)',      0.221, 0.842, False, True),
    ('9L2\n(no-skip, L2)',          0.219, 0.336, False, False),
    ('9b\n(no-skip, L3+L4)',        0.301, 0.559, False, False),
    ('9a\n(no-skip, L4)',           0.312, 0.606, False, False),
    ('9L3\n(no-skip, L3)',          0.319, 0.554, False, False),
    ('Baseline\nU-Net',             0.349, 0.823, True,  None),
]
data_sorted = sorted(data, key=lambda x: x[1])  # sort by AP

labels = [d[0] for d in data_sorted]
ap_vals = [d[1] for d in data_sorted]
dice_vals = [d[2] for d in data_sorted]
is_unet = [d[3] for d in data_sorted]
is_skip = [d[4] for d in data_sorted]

colors = []
for unet, skip in zip(is_unet, is_skip):
    if unet:
        colors.append('#2CA02C')   # green for U-Net
    elif skip:
        colors.append('#5B9BD5')   # blue for skip
    else:
        colors.append('#ED7D31')   # orange for no-skip

y = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(8.5, 5.5))

bars = ax.barh(y, ap_vals, color=colors, alpha=0.88, edgecolor='white', linewidth=0.5, zorder=3)

# U-Net reference line
unet_ap = 0.349
ax.axvline(unet_ap, color='#2CA02C', linestyle='--', lw=1.8, alpha=0.85, zorder=4)
ax.text(unet_ap + 0.002, len(labels) - 0.5, f'U-Net AP = {unet_ap:.3f}',
        color='#2CA02C', fontsize=9.5, va='center', fontweight='bold')

# Annotate % of U-Net AP
for i, (val, unet, skip) in enumerate(zip(ap_vals, is_unet, is_skip)):
    pct = val / unet_ap * 100
    label = f'{val:.3f}  ({pct:.0f}%)' if not unet else f'{val:.3f}  (100%)'
    ax.text(val + 0.004, i, label, va='center', fontsize=9)

# 86% and 15% callout lines
ax.axvline(0.301, color='#ED7D31', linestyle=':', lw=1.2, alpha=0.6)
ax.axvline(0.057, color='#5B9BD5', linestyle=':', lw=1.2, alpha=0.6)

ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=9.5)
ax.set_xlabel('Average Precision (AP)', fontsize=11)
ax.set_xlim(0, 0.41)
ax.set_title('Fig 4  |  AP vs U-Net Upper Bound — No-skip prototypes reach 86–89%',
             fontsize=12, fontweight='bold', pad=10)
ax.grid(axis='x', alpha=0.3, zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legend
legend_handles = [
    mpatches.Patch(color='#5B9BD5', label='With skip (ProtoSegNet)'),
    mpatches.Patch(color='#ED7D31', label='No skip (ProtoSegNetV2)'),
    mpatches.Patch(color='#2CA02C', label='Baseline U-Net (upper bound)'),
]
ax.legend(handles=legend_handles, fontsize=9.5, loc='lower right')

fig.tight_layout()
out_path = FIG_DIR / 'fig4_ap_vs_unet.png'
fig.savefig(out_path, bbox_inches='tight', dpi=150)
print(f'Saved: {out_path}')
plt.close()

Saved: report/v10/figures/fig4_ap_vs_unet.png


## Summary

In [5]:
figs = list(FIG_DIR.glob('fig*.png'))
print(f'{len(figs)} figures saved to {FIG_DIR}:')
for f in sorted(figs):
    size_kb = f.stat().st_size // 1024
    print(f'  {f.name:40s}  {size_kb} KB')

3 figures saved to report/v10/figures:
  fig2_2x2_heatmap.png                      91 KB
  fig3_pixel_vs_patch_faith.png             75 KB
  fig4_ap_vs_unet.png                       99 KB
